# Modèle final — entraînement, évaluation et interprétation

**Modèle retenu** (voir `02_modeling`) : régression logistique équilibrée, régularisation
*elastic net*, jeu de variables complet, seuil maximisant le F1 sur les probabilités
out-of-fold. Sa configuration est dans `configs/config.yaml`.

Ce notebook ne contient aucune logique de modélisation : il appelle les mêmes
fonctions que les commandes

```
python -m bank_marketing.application.train
python -m bank_marketing.application.evaluate
```

et documente les résultats.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from bank_marketing.application.evaluate import run_evaluation
from bank_marketing.application.preparation import prepare_train_test
from bank_marketing.application.train import DEFAULT_MODEL_PATH, run_training
from bank_marketing.domain.evaluation import build_confusion_table
from bank_marketing.domain.interpretation import (
    extract_logistic_coefficients,
    fit_inference_model,
)
from bank_marketing.domain.preprocessing import select_features
from bank_marketing.infrastructure.artifacts import load_model
from bank_marketing.infrastructure.config import load_config
from bank_marketing.infrastructure.figures import save_figure
from bank_marketing.settings import DEFAULT_CONFIG_PATH

sns.set_theme(style="whitegrid")
config = load_config(DEFAULT_CONFIG_PATH)
config["model"]

## 1. Entraînement et choix du seuil

Le modèle est entraîné sur le jeu d'entraînement. Le seuil est choisi sur ses
probabilités out-of-fold : **le jeu de test n'intervient pas**.

In [ ]:
train_results = run_training(config)
pd.Series(train_results, name="OOF (entraînement)").round(3)

## 2. Évaluation unique sur le jeu de test

In [ ]:
test_results = run_evaluation(config)

pd.DataFrame(
    {"OOF (entraînement)": train_results, "Test": test_results}
).round(3)

Les performances du test sont très proches des estimations out-of-fold
(F1 : 0,485 contre 0,487 ; PR-AUC : 0,420 contre 0,423). Le modèle **ne surapprend
pas** et le seuil choisi sur l'entraînement se transfère correctement.

In [ ]:
model, threshold = load_model(DEFAULT_MODEL_PATH)
_, x_test, _, y_test = prepare_train_test(config)
confusion = build_confusion_table(y_test, model.predict_proba(x_test)[:, 1], threshold)

figure, axis = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axis)
axis.set_title(f"Matrice de confusion du test (seuil = {threshold:.3f})")
save_figure(figure, "matrice_confusion_test")
plt.show()

**Lecture métier :** en appelant uniquement les 108 clients ciblés par le modèle
(13 % du jeu de test), la banque atteindrait 48 des 90 souscripteurs (53 %), avec
une précision de 44 %, soit environ **4 fois** le taux de souscription d'un appel au
hasard (≈ 11 %).

## 3. Coefficients du modèle de prédiction

Les variables numériques étant standardisées, un **odds ratio** (OR) se lit :

- OR > 1 : la variable **augmente** les chances de souscription ;
- OR < 1 : elle les **diminue** ;
- pour une variable numérique, l'effet correspond à une hausse d'un écart-type.

La pénalité réduit volontairement les coefficients vers zéro : leur ampleur est
atténuée, mais leur sens et leur ordre d'importance restent interprétables.

In [ ]:
coefficients = extract_logistic_coefficients(model)
coefficients.head(15)

In [ ]:
top_coefficients = coefficients.head(15).sort_values("coefficient")

figure, axis = plt.subplots(figsize=(8, 6))
colors = ["#2A9D8F" if value > 0 else "#E76F51" for value in top_coefficients["coefficient"]]
axis.barh(top_coefficients.index, top_coefficients["coefficient"], color=colors)
axis.axvline(0, color="grey", linewidth=1)
axis.set(
    title="15 coefficients les plus importants (régression logistique)",
    xlabel="Coefficient (> 0 : favorise la souscription)",
)
save_figure(figure, "coefficients_modele_final")
plt.show()

## 4. Significativité des effets

scikit-learn ne fournit pas de p-values, et celles-ci ne sont pas valides pour un
modèle pénalisé. Une régression logistique **non pénalisée** est donc estimée avec
`statsmodels`, sur le jeu de variables interprétable, à des fins d'**interprétation
uniquement**. Pour la rendre estimable :

- `loan` est retirée (ses `unknown` sont exactement ceux de `housing`) ;
- `default` est retirée (modalité `yes` quasi absente) ;
- `previously_contacted` est retirée (information déjà portée par `poutcome`) ;
- la modalité rare `illiterate` d'`education` est regroupée avec `basic.4y`.

Modalités de référence : avril (`month`), admin. (`job`), divorcé (`marital`) et
échec lors de la campagne précédente (`poutcome`).

In [ ]:
x_train, _, y_train, _ = prepare_train_test(config)
significance = fit_inference_model(select_features(x_train, "interpretable"), y_train)
significance[significance["significatif_5pct"]]

| Effet | OR | Interprétation |
|---|---|---|
| `euribor3m` | 0,48 | Des taux d'intérêt élevés réduisent fortement les souscriptions |
| `poutcome_success` | 5,50 | Un client ayant souscrit lors d'une campagne précédente a environ 5,5 fois plus de chances (en odds) de souscrire qu'un client ayant refusé |
| `poutcome_nonexistent` | 2,64 | Un client jamais contacté souscrit plus qu'un client ayant déjà refusé |
| `previous` | 1,37 | Les contacts lors des campagnes passées augmentent les chances de souscription |
| `month_mar`, `month_dec`, `month_jun` | 4,92 ; 4,20 ; 2,70 | Ces mois convertissent mieux qu'avril |
| `campaign` | 0,80 | Multiplier les appels pendant la campagne réduit les chances (lassitude) |
| `job_management` | 0,54 | Les cadres souscrivent moins que les employés administratifs |
| `age` | 1,18 | Les chances augmentent légèrement avec l'âge |

Les effets majeurs (Euribor, historique de campagne, mois, nombre d'appels) vont dans
le même sens dans le modèle de prédiction, ce qui conforte leur interprétation.

**Précautions :**

- ces résultats décrivent des **associations, pas des causes** : les effets du mois
  et de l'Euribor reflètent surtout le contexte de chaque campagne (données
  2008-2010, pendant la crise financière) ;
- environ 40 tests sont réalisés : au seuil de 5 %, quelques résultats significatifs
  peuvent être dus au hasard. Les effets avec p < 0,01 sont les plus fiables ;
- les effets de mars et de décembre reposent sur peu de contacts, d'où des
  intervalles de confiance larges.

## 5. Conclusion et recommandations

Le modèle final cible les clients environ **4 fois mieux qu'un appel au hasard**, avec
des performances stables entre validation croisée et test. Sa nature linéaire permet
d'en tirer des recommandations concrètes :

1. **Prioriser les clients ayant déjà souscrit** lors d'une campagne précédente :
   c'est le levier le plus fort et le plus fiable.
2. **Limiter les relances** auprès d'un même client pendant une campagne.
3. **Tenir compte du contexte de taux** : les campagnes de dépôts sont moins efficaces
   quand les taux sont élevés.

Le seuil de décision (0,589) peut être ajusté selon le coût d'un appel : un seuil
plus bas détecte davantage de souscripteurs au prix de plus d'appels inutiles.